Notebook 1: Configuração do ambiente e carregamento do dataset

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np

arquivo_nome = "ventiladores-de-mesa-de-coluna-e-circulador-de-ar.csv"

diretorio_atual = Path.cwd()

caminhos_encontrados = list(diretorio_atual.rglob(arquivo_nome))

if not caminhos_encontrados:
    caminhos_encontrados = list(diretorio_atual.parent.rglob(arquivo_nome))

if caminhos_encontrados:
    caminho_final = caminhos_encontrados[0]
    print(f"Arquivo localizado com sucesso em: {caminho_final}")

    df = pd.read_csv(caminho_final, encoding="utf-8-sig")

    df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('°C', 'C').str.replace('/', '_')

    print("Dataset carregado e colunas padronizadas!")
    print(f"Linhas brutas: {len(df)}")
else:
    caminho_final = Path("../Datasets Itens de casa/" + arquivo_nome)
    df = pd.read_csv(caminho_final, encoding="utf-8-sig")
    df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('°C', 'C').str.replace('/', '_')
    print("Carregado via caminho manual!")

Arquivo localizado com sucesso em: /home/claude/work/ventiladores-de-mesa-de-coluna-e-circulador-de-ar.csv
Dataset carregado e colunas padronizadas!
Linhas brutas: 5178


Notebook 2: Seleção de atributos (Feature selection)

In [2]:
if 'Status_do_Modelo' in df.columns:
    antes = len(df)
    df = df[df['Status_do_Modelo'] != 'Excluido']
    print(f"✓ Registros de modelos excluídos removidos: {antes - len(df)}")

colunas_para_remover = [
    'Código_de_Barras',
    'Status_do_Modelo',
    'Status_do_Registro',
    'Número_do_Registro'
]

colunas_existentes = [col for col in colunas_para_remover if col in df.columns]
df = df.drop(columns=colunas_existentes)

print(f"✓ {len(colunas_existentes)} colunas irrelevantes removidas.")
print(f"Colunas restantes ({len(df.columns)}):", list(df.columns))

✓ Registros de modelos excluídos removidos: 274
✓ 4 colunas irrelevantes removidas.
Colunas restantes (25): ['Marca', 'Modelo', 'Tipo_de_Produto', 'Tensão', 'Tipo_de_Controle', 'Diâmetro_da_Hélice_cm', 'No._de_Pás', 'Vazão_Média_de_Ar_m3_h_Baixa', 'Vazão_Média_de_Ar_m3_h_Média', 'Vazão_Média_de_Ar_m3_h_Alta', 'Potência_Elétrica_Consumida_W_Baixa', 'Potência_Elétrica_Consumida_W_Média', 'Potência_Elétrica_Consumida_W_Alta', 'Eficiência_m3_s_W_Baixa', 'Eficiência_m3_s_W_Média', 'Eficiência_m3_s_W_Alta', 'Consumo_de_Energia_MensalkWh_mês_Baixa', 'Consumo_de_Energia_MensalkWh_mês_Média', 'Consumo_de_Energia_MensalkWh_mês_Alta', 'Classificação_Baixa', 'Classificação_Média', 'Classificação_Alta', 'Selo_Procel_Baixa', 'Selo_Procel_Média', 'Selo_Procel_Alta']


Notebook 3: Padronização de textos, categorias e unidades

In [3]:
def sem_acento(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii').lower()

def classificar_tipo(texto):
    if pd.isna(texto):
        return None
    t = sem_acento(str(texto))
    if 'hibrid' in t or '3 em 1' in t or '2 em 1' in t:
        return 'Híbrido'
    encontrados = set()
    if re.search(r'circu', t):
        encontrados.add('Circulador')
    if re.search(r'\bmesa\b|\bmea\b', t):
        encontrados.add('Mesa')
    if re.search(r'parede', t):
        encontrados.add('Parede')
    if re.search(r'pedest|pedes?tr|coluna', t):
        encontrados.add('Pedestal')
    if len(encontrados) > 1:
        return 'Híbrido'
    if len(encontrados) == 1:
        return encontrados.pop()
    return None

df['Marca'] = df['Marca'].astype(str).str.strip().str.upper()
df['Modelo'] = df['Modelo'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df.loc[df['Modelo'].str.lower() == 'nan', 'Modelo'] = np.nan

tipo_original = df['Tipo_de_Produto'].apply(classificar_tipo)
tipo_pelo_modelo = df['Modelo'].apply(classificar_tipo)
df['Tipo_de_Produto'] = tipo_original.fillna(tipo_pelo_modelo).fillna('Não informado')

df['Tensão'] = df['Tensão'].astype(str).str.strip().str.upper().str.replace(' ', '')
df['Tensão'] = df['Tensão'].replace({'127V': '127 V', '220V': '220 V', 'BIVOLT': 'Bivolt', 'NAN': np.nan})

def tensao_pelo_modelo(texto):
    if pd.isna(texto):
        return np.nan
    t = str(texto).upper().replace(' ', '')
    if 'BIV' in t or '127/220' in t or '127V/220V' in t or '220/127' in t or '110/220' in t:
        return 'Bivolt'
    tem_127 = bool(re.search(r'127V|110V', t))
    tem_220 = bool(re.search(r'220V', t))
    if tem_127 and not tem_220:
        return '127 V'
    if tem_220 and not tem_127:
        return '220 V'
    return np.nan

def diametro_pelo_modelo(texto):
    if pd.isna(texto):
        return np.nan
    m = re.search(r'(?<![\d.,])(\d{2})\s?cm', str(texto).lower())
    if m and 25 <= float(m.group(1)) <= 70:
        return float(m.group(1))
    return np.nan

df['Tensão'] = df['Tensão'].fillna(df['Modelo'].apply(tensao_pelo_modelo))

df.loc[df['Diâmetro_da_Hélice_cm'] > 100, 'Diâmetro_da_Hélice_cm'] = df['Diâmetro_da_Hélice_cm'] / 10
df.loc[~df['Diâmetro_da_Hélice_cm'].between(10, 100), 'Diâmetro_da_Hélice_cm'] = np.nan
df['Diâmetro_da_Hélice_cm'] = df['Diâmetro_da_Hélice_cm'].fillna(df['Modelo'].apply(diametro_pelo_modelo))
df.loc[~df['Diâmetro_da_Hélice_cm'].between(10, 100), 'Diâmetro_da_Hélice_cm'] = np.nan

df.loc[df['No._de_Pás'] == 0, 'No._de_Pás'] = np.nan

for nivel in ['Baixa', 'Média', 'Alta']:
    col = f'Eficiência_m3_s_W_{nivel}'
    mascara = (df[col] >= 1) & (df[col] == df[col].round())
    df.loc[mascara, col] = df.loc[mascara, col] / 10000
    df.loc[df[col] > 0.05, col] = np.nan

print("✓ Marca, Modelo, Tipo, Tensão, diâmetro (mm → cm) e eficiência padronizados.")
print(df['Tipo_de_Produto'].value_counts().to_string())

✓ Marca, Modelo, Tipo, Tensão, diâmetro (mm → cm) e eficiência padronizados.
Tipo_de_Produto
Não informado    1443
Pedestal         1277
Mesa             1052
Parede            747
Híbrido           273
Circulador        112


Notebook 4: Tratamento de Duplicatas e Registros Redundantes

In [4]:
linhas_antes = len(df)
df = df.drop_duplicates()
print(f"✓ Linhas duplicadas exatas removidas: {linhas_antes - len(df)}")

if 'Modelo' in df.columns:
    df['contagem_nulos'] = df.isnull().sum(axis=1)
    df['chave_modelo'] = df['Marca'] + '|' + df['Modelo'].fillna('').str.upper()
    linhas_antes = len(df)
    df = df.sort_values('contagem_nulos').drop_duplicates(subset=['chave_modelo'], keep='first')
    df = df.drop(columns=['contagem_nulos', 'chave_modelo'])
    print(f"✓ Modelos duplicados tratados: {linhas_antes - len(df)} removidos. Total de linhas atual: {len(df)}")

✓ Linhas duplicadas exatas removidas: 514
✓ Modelos duplicados tratados: 202 removidos. Total de linhas atual: 4188


Notebook 5: Filtragem de Erros e Outliers Irrealistas.

In [5]:
potencias = [f'Potência_Elétrica_Consumida_W_{n}' for n in ['Baixa', 'Média', 'Alta']]
consumos = [f'Consumo_de_Energia_MensalkWh_mês_{n}' for n in ['Baixa', 'Média', 'Alta']]
vazoes = [f'Vazão_Média_de_Ar_m3_h_{n}' for n in ['Baixa', 'Média', 'Alta']]

linhas_antes = len(df)
df = df.dropna(subset=['Potência_Elétrica_Consumida_W_Alta'])
print(f"✓ Linhas sem dados técnicos removidas: {linhas_antes - len(df)}")

df = df[df['Potência_Elétrica_Consumida_W_Alta'].between(10, 250)]

ordem_invalida = (
    (df['Potência_Elétrica_Consumida_W_Baixa'] > df['Potência_Elétrica_Consumida_W_Alta']) |
    (df['Potência_Elétrica_Consumida_W_Média'] > df['Potência_Elétrica_Consumida_W_Alta'])
)
df = df[~ordem_invalida]

for p in potencias[:2]:
    df.loc[df[p] < 3, p] = np.nan

for v in vazoes:
    df.loc[(df[v] > 5) | (df[v] <= 0), v] = np.nan

for v in vazoes:
    q1 = df.groupby('Tipo_de_Produto')[v].transform(lambda x: x.quantile(0.25))
    q3 = df.groupby('Tipo_de_Produto')[v].transform(lambda x: x.quantile(0.75))
    limite = q3 + 3 * (q3 - q1)
    df.loc[df[v] > limite, v] = np.nan

for p, c in zip(potencias, consumos):
    esperado = df[p] * 0.03
    divergente = (df[c] - esperado).abs() > (esperado * 0.10)
    df.loc[divergente, c] = np.nan

print(f"✓ Filtros de inconsistência aplicados. Linhas restantes: {len(df)}")

✓ Linhas sem dados técnicos removidas: 1446
✓ Filtros de inconsistência aplicados. Linhas restantes: 2621


Notebook 6: Imputação Inteligente e Exportação dos Dados Limpos.

In [6]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier

df = df[df['Tipo_de_Produto'] != 'Não informado']
df = df.dropna(subset=['Tensão']).reset_index(drop=True)

warnings.filterwarnings('ignore', category=ConvergenceWarning)

niveis = ['Baixa', 'Média', 'Alta']
eficiencias = [f'Eficiência_m3_s_W_{n}' for n in niveis]
classificacoes = [f'Classificação_{n}' for n in niveis]
colunas_numericas = ['Diâmetro_da_Hélice_cm', 'No._de_Pás'] + potencias + vazoes + eficiencias

df['Controle_estimado'] = df['Tipo_de_Controle'].isnull()
df['Diametro_Pas_estimado'] = df[['Diâmetro_da_Hélice_cm', 'No._de_Pás']].isnull().any(axis=1)
df['Potencia_estimada'] = df[potencias[:2]].isnull().any(axis=1)
df['Vazao_estimada'] = df[vazoes].isnull().any(axis=1)
df['Eficiencia_estimada'] = df[eficiencias].isnull().any(axis=1)
df['Classificacao_estimada'] = df[classificacoes].isnull().any(axis=1)

moda_controle = df.groupby('Tipo_de_Produto')['Tipo_de_Controle'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
df['Tipo_de_Controle'] = df['Tipo_de_Controle'].fillna(moda_controle)

categoricas = pd.get_dummies(df[['Tipo_de_Produto', 'Tensão', 'Tipo_de_Controle']], dtype=float)
matriz = pd.concat([df[colunas_numericas], categoricas], axis=1)

imputador = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=30, min_samples_leaf=2, random_state=42, n_jobs=-1),
    max_iter=4,
    initial_strategy='median',
    random_state=42
)
imputado = pd.DataFrame(imputador.fit_transform(matriz), columns=matriz.columns, index=matriz.index)
df[colunas_numericas] = imputado[colunas_numericas]

df['No._de_Pás'] = df['No._de_Pás'].round()
df['Diâmetro_da_Hélice_cm'] = df['Diâmetro_da_Hélice_cm'].round(1)

df['Potência_Elétrica_Consumida_W_Média'] = df[['Potência_Elétrica_Consumida_W_Média', 'Potência_Elétrica_Consumida_W_Alta']].min(axis=1)
df['Potência_Elétrica_Consumida_W_Baixa'] = df[['Potência_Elétrica_Consumida_W_Baixa', 'Potência_Elétrica_Consumida_W_Média']].min(axis=1)
df['Vazão_Média_de_Ar_m3_h_Média'] = df[['Vazão_Média_de_Ar_m3_h_Média', 'Vazão_Média_de_Ar_m3_h_Alta']].min(axis=1)
df['Vazão_Média_de_Ar_m3_h_Baixa'] = df[['Vazão_Média_de_Ar_m3_h_Baixa', 'Vazão_Média_de_Ar_m3_h_Média']].min(axis=1)

for p, c in zip(potencias, consumos):
    esperado = df[p] * 0.03
    df.loc[(df[c] - esperado).abs() > esperado * 0.10, c] = np.nan
    df[c] = df[c].fillna(esperado.round(2))

for n, alvo in zip(niveis, classificacoes):
    atributos = [f'Eficiência_m3_s_W_{n}', f'Vazão_Média_de_Ar_m3_h_{n}', f'Potência_Elétrica_Consumida_W_{n}', 'Diâmetro_da_Hélice_cm']
    conhecidos = df[alvo].notna()
    if (~conhecidos).any():
        modelo = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
        modelo.fit(df.loc[conhecidos, atributos], df.loc[conhecidos, alvo].astype(int))
        df.loc[~conhecidos, alvo] = modelo.predict(df.loc[~conhecidos, atributos])

df = df.dropna(subset=['Tipo_de_Controle'] + potencias + consumos + vazoes)

colunas_inteiras = ['No._de_Pás'] + classificacoes + [f'Selo_Procel_{n}' for n in niveis]
df[colunas_inteiras] = df[colunas_inteiras].astype('Int64')

df = df.sort_values(['Marca', 'Modelo']).reset_index(drop=True)

print(f"✓ Imputação multivariada concluída. Linhas prontas para uso: {len(df)}")

✓ Imputação multivariada concluída. Linhas prontas para uso: 2371


Notebook 7: Relatório de qualidade e exportação

In [7]:
colunas_flag = ['Controle_estimado', 'Diametro_Pas_estimado', 'Potencia_estimada', 'Vazao_estimada', 'Eficiencia_estimada', 'Classificacao_estimada']
print("Percentual de linhas com algum valor estimado:")
print((df[colunas_flag].mean() * 100).round(1).astype(str).add('%').to_string())

nulos = df.isnull().sum()
print("\nValores ainda vazios:")
print(nulos[nulos > 0].to_string())

print(f"\nLinhas duplicadas (Marca + Modelo): {df.duplicated(['Marca', 'Modelo']).sum()}")
print(f"Linhas 100% originais (sem nenhum valor estimado): {(~df[colunas_flag].any(axis=1)).sum()} de {len(df)}")

caminho_saida = caminho_final.parent / "ventiladores_de_mesa_limpo.csv"
df.to_csv(caminho_saida, index=False, encoding="utf-8-sig")
print(f"\n✓ Dataset limpo salvo com sucesso em: {caminho_saida}")

Percentual de linhas com algum valor estimado:
Controle_estimado          4.9%
Diametro_Pas_estimado      7.3%
Potencia_estimada          0.7%
Vazao_estimada            17.0%
Eficiencia_estimada       14.2%
Classificacao_estimada    13.5%

Valores ainda vazios:
Selo_Procel_Baixa    329
Selo_Procel_Média    335
Selo_Procel_Alta     325

Linhas duplicadas (Marca + Modelo): 0
Linhas 100% originais (sem nenhum valor estimado): 1893 de 2371

✓ Dataset limpo salvo com sucesso em: /home/claude/work/ventiladores_de_mesa_limpo.csv
